# 01 — Data Preparation

Prepare input data for k-means clustering evaluation.

**Prerequisites:** Central batch correction must already be done. The corrected
matrices live in `evaluation_data/<dataset>/after/` and are produced by:
- **Proteomics:** `evaluation_data/proteomics/01_data_prep_and_central_RBE.ipynb`
- **Microarray:** `evaluation_data/microarray/02_central_RBE.ipynb`
- **Proteomics multibatch:** `evaluation_data/proteomics_multibatch/01_data_prep_RBE.ipynb`
- **SCP protein scenario 2:** `evaluation_data/scp_protein/01_data_prep_protein_level.ipynb`

These R notebooks run limma `removeBatchEffect` and save the corrected
expression matrices. This notebook does NOT re-run the correction — it only loads
the results.

**What this notebook does:**
1. Load raw (before correction) and corrected expression matrices from `evaluation_data/`.
2. Align sample columns to metadata and filter NaN feature rows (intersection, because k-means implementation used here requires complete data).
3. Save prepared (filtered, aligned) matrices to `real_datasets/<dataset>/prepared/`
   for downstream notebooks (02–04).

## Imports and Setup

In [5]:
import sys
from pathlib import Path
import pandas as pd

# Resolve paths relative to this notebook
NOTEBOOK_DIR = Path.cwd()  # evaluation_clusterization_after_correction/real_datasets/
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    discover_clients,
    load_metadata,
    load_feature_matrix,
    align_matrix_to_metadata,
    drop_rows_with_any_na,
    choose_corrected_path,
    load_matrix_with_lfs_fallback,
)

## Configuration

Select which datasets to prepare. Add new datasets by extending
`dataset_configs()` in `evaluation_utils/real_datasets_utils.py`.

In [6]:
# Datasets to prepare. Available: "proteomics", "microarray", "proteomics_multibatch", "ccRCC_proteomics".
DATASETS = [
    "proteomics", 
    "microarray", 
    "proteomics_multibatch", "ccRCC_proteomics"
    ]

# Which corrected matrix to use: "central", "federated", or "auto" (prefer federated).
CORRECTED_SOURCE = "central"

# Drop feature rows containing any NA? Required for k-means on UNION matrices.
REMOVE_NA = True

# Output directory for prepared intermediate files (alongside this notebook).
OUTPUT_ROOT = NOTEBOOK_DIR
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Load and Prepare Data

For each dataset: load before/corrected matrices from `evaluation_data/`,
align to metadata, and filter NaN rows. Uses `load_matrix_with_lfs_fallback()`
to handle unfetched Git LFS pointers automatically.

In [7]:
# Store prepared data for use by downstream notebooks.
# Keys: dataset name -> dict with 'before', 'corrected', 'metadata', 'k_values', etc.
prepared = {}

configs = dataset_configs(REPO_ROOT)

for ds_name in DATASETS:
    cfg = configs[ds_name]
    print(f"\n{'='*60}")
    print(f"Preparing: {ds_name}")
    print(f"{'='*60}")

    # Discover sites and load metadata
    clients = discover_clients(cfg)
    metadata = load_metadata(cfg, clients)
    print(f"Sites: {[c.name for c in clients]}")
    print(f"Samples: {metadata.shape[0]}, Conditions: {metadata['condition'].nunique()}, "
          f"Batches: {metadata['lab'].nunique()}")

    # Load before matrix (with LFS fallback)
    before_raw = load_matrix_with_lfs_fallback(cfg.before_matrix, f"{ds_name} before", cfg)
    before = align_matrix_to_metadata(before_raw, metadata, f"{ds_name} before")
    print(f"Before matrix: {before.shape[0]} features x {before.shape[1]} samples")

    # Load corrected matrix (with LFS fallback)
    corr_path, corr_kind = choose_corrected_path(cfg, CORRECTED_SOURCE)
    corrected_raw = load_matrix_with_lfs_fallback(corr_path, f"{ds_name} corrected", cfg)
    corrected = align_matrix_to_metadata(corrected_raw, metadata, f"{ds_name} corrected")
    print(f"Corrected matrix ({corr_kind}): {corrected.shape[0]} features x {corrected.shape[1]} samples")

    # Filter NA if requested
    if REMOVE_NA:
        before = drop_rows_with_any_na(before, f"{ds_name} before")
        corrected = drop_rows_with_any_na(corrected, f"{ds_name} corrected")

    # Determine k values from data structure
    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))
    print(f"k values: {k_values} (condition={k_condition}, batch={k_batch})")

    # Save to output directory for other notebooks
    ds_out = OUTPUT_ROOT / ds_name / "prepared"
    ds_out.mkdir(parents=True, exist_ok=True)
    before.to_csv(ds_out / "before_matrix.tsv", sep="\t")
    corrected.to_csv(ds_out / "corrected_matrix.tsv", sep="\t")
    metadata.to_csv(ds_out / "metadata.tsv", sep="\t", index=False)
    print(f"Saved prepared data to {ds_out}")

    prepared[ds_name] = {
        'before': before,
        'corrected': corrected,
        'metadata': metadata,
        'clients': clients,
        'k_condition': k_condition,
        'k_batch': k_batch,
        'k_values': k_values,
    }


Preparing: proteomics
Sites: ['lab_A', 'lab_B', 'lab_C', 'lab_D', 'lab_E']
Samples: 118, Conditions: 2, Batches: 5
Before matrix: 3059 features x 118 samples
Corrected matrix (central): 2702 features x 118 samples
[proteomics before] remove_na=True dropped 1002 of 3059 feature rows containing NA
[proteomics corrected] remove_na=True dropped 645 of 2702 feature rows containing NA
k values: [2, 5] (condition=2, batch=5)
Saved prepared data to /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics/prepared

Preparing: microarray
Sites: ['GSE14407', 'GSE26712', 'GSE38666', 'GSE40595', 'GSE6008', 'GSE69428']
Samples: 332, Conditions: 2, Batches: 6
Before matrix: 51276 features x 332 samples
Corrected matrix (central): 51276 features x 332 samples
[microarray before] remove_na=True dropped 30148 of 51276 feature rows containing NA
[microarray corrected] remove_na=True dropped 30148 of 51276 feature rows containing NA
k values: [2, 6] (c

## Summary

In [8]:
# Print summary table of prepared datasets.
summary = []
for ds_name, data in prepared.items():
    summary.append({
        'Dataset': ds_name,
        'Samples': data['metadata'].shape[0],
        'Features (before)': data['before'].shape[0],
        'Features (corrected)': data['corrected'].shape[0],
        'Conditions': data['k_condition'],
        'Batches': data['k_batch'],
        'k values': str(data['k_values']),
    })
pd.DataFrame(summary)

,Dataset,Samples,Features (before),Features (corrected),Conditions,Batches,k values
0,proteomics,118,2057,2057,2,5,"[2, 5]"
1,microarray,332,21128,21128,2,6,"[2, 6]"
2,proteomics_multibatch,72,1388,1388,4,4,[4]
3,ccRCC_proteomics,887,2074,2074,2,3,"[2, 3]"
